# Übung 4: Entscheidungsbäume für die Extruder-Blockade
**Internet of Production – KI, Woche 2** · ca. 40 min · Arbeit in Zweiergruppen

**Roter Faden:** In Woche 1 haben Sie einen KNN-Klassifikator gebaut, der vor einer Extruder-Blockade warnt (Übung 2). Im ersten Block dieser Woche haben Sie ihn in KNIME nachgebaut. Jetzt suchen Sie ein Modell, das **leicht zu lesen und der Werkstatt zu erklären** ist: einen Entscheidungsbaum.

**Fragen dieser Übung**
1. Reicht ein einzelner Schwellwert auf der Energie?
2. Wie tief darf ein Baum werden, bevor er die Trainingsdaten nur auswendig lernt?
3. Lässt sich der Baum als Regel lesen, die ein Techniker versteht?
4. Wie schneidet er im Vergleich zu KNN ab?

Jedes `___` ist eine Lücke. Der Code zur Datenaufbereitung ist Übung 2 in einer Funktion; Sie müssen ihn nicht ändern.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

DATA = Path('../../00_Data/printfarm')
HORIZON = 360                       # minutes: warn 6 hours ahead
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(k, horizon=HORIZON):
    """The pipeline of exercise 2 of week 1 in one function: one row per 10th printing minute."""
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    blockages = np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values)
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages, idx)
    mtb = np.where(nxt < len(blockages), (blockages[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    p['Temp_Extruder_m'] = p['Temp_Extruder'].rolling(120).mean()
    p['Temp_Bed_m'] = p['Temp_Bed'].rolling(120).mean()
    return p.dropna(subset=FEATURES + ['Temp_Extruder_m']).iloc[::10], pd.to_datetime(blockages)

def report(name, y_true, y_pred, score=None):
    auc = f'   AUC {roc_auc_score(y_true, score):5.3f}' if score is not None else ''
    print(f'{name:36s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}{auc}')

data, blockages = make_blockage_data(1)
cut = int(len(data) * 0.7)                      # split by time: first 70 % train, last 30 % test
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train, X_test, y_test = train[FEATURES], train['y'], test[FEATURES], test['y']
print(len(data), 'samples,', f"{data['y'].mean():.1%} class 1; train {len(train)}, test {len(test)}")

## Schritt 1: Eine Regel von Hand
Stellen Sie die Verteilung von `E120` (mittlerer Energieverbrauch der letzten 2 Stunden) für beide Klassen in den **Trainingsdaten** dar. Wo würden Sie einen Schwellwert setzen?

In [ ]:
lo, hi = np.percentile(train['E120'], [0.5, 99.5])
bins = np.linspace(lo, hi, 60)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(train.loc[train['y'] == 0, 'E120'], bins=bins, density=True, alpha=.6, color='#009EE0', label='no blockage within 6 h')
ax.hist(train.loc[train['y'] == 1, 'E120'], bins=bins, density=True, alpha=.6, color='#EE7F00', label='blockage within 6 h')
ax.set_xlabel('E120 (Wh/min)'); ax.set_ylabel('density'); ax.legend(); plt.show()

In [ ]:
THRESH = 1.68                                   # your choice from the plot (Wh/min)
rule = (test['E120'] > THRESH).astype(int)
report(f'rule: E120 > {THRESH}', y_test, rule, test['E120'])

## Schritt 2: Den Baum den Schwellwert finden lassen
Ein Baum der Tiefe 1 (ein *Stumpf*) sucht den besten einzelnen Schwellwert selbst. Findet er Ihren?

In [ ]:
stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train)
print(export_text(stump, feature_names=FEATURES, decimals=3))
report('decision stump', y_test, stump.predict(X_test), stump.predict_proba(X_test)[:, 1])

## Schritt 3: Wie tief darf der Baum sein?
Trainieren Sie Bäume zunehmender Tiefe (`None` = wachsen, bis jedes Blatt rein ist). Vergleichen Sie das Ergebnis auf den **Trainingsdaten** mit dem auf den **Testdaten**.

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 12, None]
rows = []
for depth in depths:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    rows.append({'depth': str(depth), 'leaves': t.get_n_leaves(),
                 'AUC train': roc_auc_score(y_train, t.predict_proba(X_train)[:, 1]),
                 'AUC test': roc_auc_score(y_test, t.predict_proba(X_test)[:, 1]),
                 'recall test': recall_score(y_test, t.predict(X_test)),
                 'precision test': precision_score(y_test, t.predict(X_test), zero_division=0)})
res = pd.DataFrame(rows).set_index('depth')
display(res.round(3))
res[['AUC train', 'AUC test']].plot(marker='o', figsize=(7, 4), ylabel='AUC'); plt.show()

**Beschreiben Sie die Kurven. Welche Tiefe wählen Sie, und warum nicht den tiefsten Baum?**

Die Trainings-AUC steigt mit der Tiefe gegen 1,0, die Test-AUC ist bei Tiefe 2 bis 4 am besten (etwa 0,87 bis 0,88) und fällt beim unbeschränkten Baum auf etwa 0,7: Er lernt einzelne verrauschte Stichproben auswendig (Überanpassung). Tiefe 2 bis 4 genügt; ein tieferer Baum bringt Komplexität, aber keine Information, weil das Signal eine kleine Verschiebung eines einzigen Merkmals ist.

## Schritt 4: Ein Baum, den man lesen kann
Beschränken Sie den Baum so, dass jedes Blatt mindestens 200 Stichproben enthält (`min_samples_leaf`), und geben Sie der seltenen Klasse mehr Gewicht (`class_weight='balanced'`). Geben Sie dann die Regeln aus.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, class_weight='balanced', random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=FEATURES, decimals=3))
fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree, feature_names=FEATURES, class_names=['ok', 'warn'], filled=True, rounded=True, fontsize=8, ax=ax); plt.show()
report('tree, depth 3, balanced', y_test, tree.predict(X_test), tree.predict_proba(X_test)[:, 1])

**Formulieren Sie die Regel des Baums in einem Satz für einen Techniker. Welches Merkmal trägt die Entscheidung?**

„Wenn der Drucker in den letzten zwei Stunden Druckzeit im Mittel mehr als etwa 1,665 Wh je Minute verbraucht hat, ist innerhalb der nächsten sechs Stunden eine Blockade wahrscheinlich. Darunter nur warnen, wenn die letzten 30 Minuten über 1,685 Wh liegen und der Trend noch steigt.“ Die Merkmalswichtigkeit (unten) zeigt, dass `E120` fast die gesamte Entscheidung trägt (etwa 96 %); `E30` und `slope` verfeinern sie nur.

In [ ]:
pd.Series(tree.feature_importances_, index=FEATURES).sort_values().plot.barh(figsize=(6, 2.5), color='#003A79', title='feature importance')
plt.show()

## Schritt 5: Der Schwellwert ist eine betriebliche Entscheidung
Verschieben Sie den Schwellwert auf `E120` und beobachten Sie Recall und Precision. Ein **niedriger** Schwellwert warnt früh und oft; ein **hoher** warnt selten, dafür verlässlich.

In [ ]:
rows = []
for th in [1.66, 1.67, 1.68, 1.69, 1.70]:
    pred = (test['E120'] > th).astype(int)
    rows.append({'threshold': th, 'recall': recall_score(y_test, pred), 'precision': precision_score(y_test, pred, zero_division=0),
                 'warned samples': int(pred.sum()), 'false alarm samples': int(((pred == 1) & (y_test == 0)).sum())})
display(pd.DataFrame(rows).set_index('threshold').round(3))

**Welchen Schwellwert empfehlen Sie, wenn die Prüfung der Düse 10 Minuten dauert und eine Blockade einen gescheiterten Auftrag und etwa 12 Stunden Stillstand kostet? Welchen, wenn der Techniker nur einmal je Schicht verfügbar ist?**

Bei billiger Prüfung und teurem Ausfall ist ein niedriger Schwellwert sinnvoll (1,66 bis 1,67: Recall 76 bis 84 %, aber nur 27 bis 50 % der Warnungen sind echt). Ist der Techniker knapp, hält ein höherer Schwellwert (1,69: Recall 55 %, Precision 78 %) die Zahl der Fehlalarme niedrig. Das Modell kann das nicht entscheiden; das Kostenverhältnis entscheidet.

## Schritt 6: Vergleich mit KNN
Alle Modelle auf demselben Testzeitraum.

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)).fit(X_train, y_train)
report('KNN, k = 25', y_test, knn.predict(X_test), knn.predict_proba(X_test)[:, 1])
report(f'rule: E120 > {THRESH}', y_test, rule, test['E120'])
report('decision stump', y_test, stump.predict(X_test), stump.predict_proba(X_test)[:, 1])
report('tree, depth 3, balanced', y_test, tree.predict(X_test), tree.predict_proba(X_test)[:, 1])

**Welches Modell geben Sie an die Werkstatt, und warum? Was haben alle Modelle gemeinsam?**

KNN, die Regel und der ausgeglichene Baum erreichen eine AUC von etwa 0,88 bis 0,91 (der Stumpf hat nur zwei Score-Stufen, seine AUC ist daher bauartbedingt niedrig); die Güte wird also vom Signal begrenzt (wenige Ereignisse, verrauschte Energie), nicht vom Algorithmus. Die Regel oder der kleine Baum sind vorzuziehen: Sie sind lesbar, vom Techniker prüfbar, ohne Modelldatei in eine SPS oder ein Monitoring übertragbar, und ihr Schwellwert lässt sich an die Kosten eines Fehlalarms anpassen. KNN braucht zur Laufzeit den gesamten Trainingsdatensatz und lässt sich nicht erklären.

## Schritt 7: Export für KNIME (schon für Sie erledigt)
Die Datei für den KNIME-Workflow wurde so erzeugt. Sie können sie für Drucker 2 neu erzeugen (für den Übertragungstest in Woche 3).

In [ ]:
def export_for_knime(k, path=None):
    d, _ = make_blockage_data(k)
    d = d.assign(label=np.where(d['y'] == 1, 'warn', 'ok'), split=np.where(np.arange(len(d)) < int(len(d) * 0.7), 'train', 'test'))
    out = d[FEATURES + ['y', 'label', 'split']].round(6).reset_index()
    out.to_csv(path or f'blockage_features_printer{k}.csv', index=False)      # comma separator, decimal point (KNIME default)
    return out

export_for_knime(1).head()